### Phần 1: Khám phá Tensor

#### Task 1.1 — Tạo Tensor

In [1]:
import torch
import numpy as np

# Tạo tensor từ list
data = [[1, 2], [3, 4]]
x_data = torch.tensor(data)
print(f"Tensor từ list:\n{x_data}\n")

# Tạo tensor từ NumPy array
np_array = np.array(data)
x_np = torch.from_numpy(np_array)
print(f"Tensor từ NumPy array:\n{x_np}\n")

# Tạo tensor gồm toàn số 1 (cùng shape)
x_ones = torch.ones_like(x_data)
print(f"Ones Tensor:\n{x_ones}\n")

# Tạo tensor ngẫu nhiên
x_rand = torch.rand_like(x_data, dtype=torch.float)
print(f"Random Tensor:\n{x_rand}\n")

# In ra thông tin Tensor
print("Shape:", x_rand.shape)
print("Dtype:", x_rand.dtype)
print("Device:", x_rand.device)


Tensor từ list:
tensor([[1, 2],
        [3, 4]])

Tensor từ NumPy array:
tensor([[1, 2],
        [3, 4]])

Ones Tensor:
tensor([[1, 1],
        [1, 1]])

Random Tensor:
tensor([[0.3078, 0.8848],
        [0.7451, 0.4022]])

Shape: torch.Size([2, 2])
Dtype: torch.float32
Device: cpu


#### Task 1.2 — Các phép toán trên Tensor

In [ ]:
# 1. Cộng x_data với chính nó
print("x_data + x_data =\n", x_data + x_data, "\n")

# 2. Nhân x_data với 5
print("x_data * 5 =\n", x_data * 5, "\n")

# 3. Nhân ma trận với chuyển vị
print("x_data @ x_data.T =\n", x_data @ x_data.T, "\n")


In [8]:
from datasets import load_dataset
dataset = load_dataset("lhoestq/conll2003")

Generating test split: 100%|██████████| 3453/3453 [00:00<00:00, 265800.39 examples/s]


In [12]:
# Tạo mapping nhãn thủ công
tag_names = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC", "B-MISC", "I-MISC"]
tag_to_ix = {tag: i for i, tag in enumerate(tag_names)}
ix_to_tag = {i: tag for i, tag in enumerate(tag_names)}

# Hàm chuyển nhãn số -> string
def convert_labels_to_str(seq_tag_ids):
    return [tag_names[i] for i in seq_tag_ids]

train_sentences = dataset["train"]["tokens"]
train_tags = [convert_labels_to_str(seq) for seq in dataset["train"]["ner_tags"]]

valid_sentences = dataset["validation"]["tokens"]
valid_tags = [convert_labels_to_str(seq) for seq in dataset["validation"]["ner_tags"]]

test_sentences = dataset["test"]["tokens"]
test_tags = [convert_labels_to_str(seq) for seq in dataset["test"]["ner_tags"]]

print(train_sentences[0])
print(train_tags[0])

['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']
['B-ORG', 'O', 'B-MISC', 'O', 'O', 'O', 'B-MISC', 'O', 'O']


In [13]:
PAD_WORD = "<PAD>"
UNK_WORD = "<UNK>"

all_words = set(word.lower() for sent in train_sentences for word in sent)
word_to_ix = {w: i+2 for i, w in enumerate(sorted(all_words))}
word_to_ix[PAD_WORD] = 0
word_to_ix[UNK_WORD] = 1

all_tags = set(tag for seq in train_tags for tag in seq)
tag_to_ix = {tag: i for i, tag in enumerate(sorted(all_tags))}

vocab_size = len(word_to_ix)
num_tags = len(tag_to_ix)

print("Vocab size:", vocab_size)
print("Num tags:", num_tags)


Vocab size: 21011
Num tags: 9


### task 2: PyTorch Dataset và DataLoader

In [14]:
import torch
from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence

PAD_TAG = -1

class NERDataset(Dataset):
    def __init__(self, sentences, tags, word_to_ix, tag_to_ix):
        self.sentences = sentences
        self.tags = tags
        self.word_to_ix = word_to_ix
        self.tag_to_ix = tag_to_ix

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        sent_idx = [self.word_to_ix.get(w.lower(), self.word_to_ix[UNK_WORD]) 
                    for w in self.sentences[idx]]
        tag_idx = [self.tag_to_ix[t] for t in self.tags[idx]]
        return torch.tensor(sent_idx, dtype=torch.long), torch.tensor(tag_idx, dtype=torch.long)


In [15]:
from torch.utils.data import DataLoader

def collate_fn(batch):
    sentences, tags = zip(*batch)
    sentences_padded = pad_sequence(sentences, batch_first=True, padding_value=word_to_ix[PAD_WORD])
    tags_padded = pad_sequence(tags, batch_first=True, padding_value=PAD_TAG)
    return sentences_padded, tags_padded

train_dataset = NERDataset(train_sentences, train_tags, word_to_ix, tag_to_ix)
valid_dataset = NERDataset(valid_sentences, valid_tags, word_to_ix, tag_to_ix)
test_dataset = NERDataset(test_sentences, test_tags, word_to_ix, tag_to_ix)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)


### Mô hình Bi-LSTM

In [16]:
import torch.nn as nn

class BiLSTMNER(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, padding_idx):
        super(BiLSTMNER, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=padding_idx)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim*2, output_dim)

    def forward(self, x):
        x_embed = self.embedding(x)
        lstm_out, _ = self.lstm(x_embed)
        out = self.fc(lstm_out)
        return out

embedding_dim = 100
hidden_dim = 128
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BiLSTMNER(vocab_size, embedding_dim, hidden_dim, num_tags, padding_idx=word_to_ix[PAD_WORD])
model.to(device)


BiLSTMNER(
  (embedding): Embedding(21011, 100, padding_idx=0)
  (lstm): LSTM(100, 128, batch_first=True, bidirectional=True)
  (fc): Linear(in_features=256, out_features=9, bias=True)
)

In [17]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_TAG)

num_epochs = 3
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for sentences, tags in train_loader:
        sentences, tags = sentences.to(device), tags.to(device)
        optimizer.zero_grad()
        outputs = model(sentences)
        loss = criterion(outputs.view(-1, num_tags), tags.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss/len(train_loader):.4f}")


Epoch 1/3, Loss: 0.5652
Epoch 2/3, Loss: 0.2612
Epoch 3/3, Loss: 0.1560


In [18]:
def evaluate(model, data_loader):
    model.eval()
    total_tokens = 0
    correct_tokens = 0
    with torch.no_grad():
        for sentences, tags in data_loader:
            sentences, tags = sentences.to(device), tags.to(device)
            outputs = model(sentences)
            predictions = torch.argmax(outputs, dim=-1)
            mask = tags != PAD_TAG
            correct_tokens += (predictions[mask] == tags[mask]).sum().item()
            total_tokens += mask.sum().item()
    return correct_tokens / total_tokens

valid_acc = evaluate(model, valid_loader)
test_acc = evaluate(model, test_loader)
print(f"Validation Accuracy: {valid_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")


Validation Accuracy: 0.9377
Test Accuracy: 0.9131


In [19]:
def predict_sentence(model, sentence, word_to_ix, tag_to_ix):
    model.eval()
    ix_to_tag = {v:k for k,v in tag_to_ix.items()}
    sentence_idx = [word_to_ix.get(word.lower(), word_to_ix[UNK_WORD]) for word in sentence]
    sentence_tensor = torch.tensor(sentence_idx, dtype=torch.long).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(sentence_tensor)
        predictions = torch.argmax(outputs, dim=-1).squeeze(0).cpu().tolist()
    return list(zip(sentence, [ix_to_tag[p] for p in predictions]))

sentence = ["VNU", "University", "is", "located", "in", "Hanoi"]
preds = predict_sentence(model, sentence, word_to_ix, tag_to_ix)
print(preds)


[('VNU', 'B-MISC'), ('University', 'I-ORG'), ('is', 'O'), ('located', 'O'), ('in', 'O'), ('Hanoi', 'O')]


In [25]:
def load_conllu(file_path):
    sentences = []
    with open(file_path, 'r', encoding='utf-8') as f:
        sentence = []
        for line in f:
            line = line.strip()
            if line == "":
                if sentence:
                    sentences.append(sentence)
                    sentence = []
            elif line.startswith("#"):
                continue  # bỏ qua comment
            else:
                parts = line.split('\t')
                if len(parts) > 4:
                    word = parts[1]
                    upos = parts[3]
                    sentence.append((word, upos))
        if sentence:
            sentences.append(sentence)
    return sentences

# Ví dụ sử dụng
train_sentences = load_conllu("../UD_English-EWT/en_ewt-ud-train.conllu")
dev_sentences = load_conllu("../UD_English-EWT/en_ewt-ud-dev.conllu")
print(f"Số câu train: {len(train_sentences)}, số câu dev: {len(dev_sentences)}")


Số câu train: 12544, số câu dev: 2001


In [26]:
from collections import Counter

# Tạo word_to_ix
word_counter = Counter(word for sent in train_sentences for word, _ in sent)
word_to_ix = {word: i for i, word in enumerate(word_counter.keys(), start=0)}
word_to_ix["<UNK>"] = len(word_to_ix)

# Tạo tag_to_ix
tags = set(tag for sent in train_sentences for _, tag in sent)
tag_to_ix = {tag: i for i, tag in enumerate(tags)}

print(f"Số từ trong từ điển: {len(word_to_ix)}, số nhãn POS: {len(tag_to_ix)}")


Số từ trong từ điển: 20201, số nhãn POS: 18


In [27]:
import torch
from torch.utils.data import Dataset

class POSDataset(Dataset):
    def __init__(self, sentences, word_to_ix, tag_to_ix):
        self.sentences = sentences
        self.word_to_ix = word_to_ix
        self.tag_to_ix = tag_to_ix

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        sentence = self.sentences[idx]
        words_idx = [self.word_to_ix.get(w, self.word_to_ix["<UNK>"]) for w, _ in sentence]
        tags_idx = [self.tag_to_ix[t] for _, t in sentence]
        return torch.tensor(words_idx, dtype=torch.long), torch.tensor(tags_idx, dtype=torch.long)


In [28]:
from torch.nn.utils.rnn import pad_sequence

PAD_IDX = -100  # dùng ignore_index cho CrossEntropyLoss

def collate_fn(batch):
    sentences, tags = zip(*batch)
    sentences_padded = pad_sequence(sentences, batch_first=True, padding_value=0)
    tags_padded = pad_sequence(tags, batch_first=True, padding_value=PAD_IDX)
    return sentences_padded, tags_padded


In [29]:
from torch.utils.data import DataLoader

train_dataset = POSDataset(train_sentences, word_to_ix, tag_to_ix)
dev_dataset = POSDataset(dev_sentences, word_to_ix, tag_to_ix)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
dev_loader = DataLoader(dev_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)


In [30]:
import torch.nn as nn

class SimpleRNNForTokenClassification(nn.Module):
    def __init__(self, vocab_size, tagset_size, embedding_dim=128, hidden_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True, bidirectional=False)
        self.fc = nn.Linear(hidden_dim, tagset_size)

    def forward(self, x):
        embeds = self.embedding(x)            # [batch, seq_len, embedding_dim]
        rnn_out, _ = self.rnn(embeds)        # [batch, seq_len, hidden_dim]
        logits = self.fc(rnn_out)            # [batch, seq_len, tagset_size]
        return logits


In [31]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SimpleRNNForTokenClassification(len(word_to_ix), len(tag_to_ix)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

def train_epoch(model, loader):
    model.train()
    total_loss = 0
    for sentences, tags in loader:
        sentences, tags = sentences.to(device), tags.to(device)
        optimizer.zero_grad()
        outputs = model(sentences)
        loss = criterion(outputs.view(-1, outputs.shape[-1]), tags.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for sentences, tags in loader:
            sentences, tags = sentences.to(device), tags.to(device)
            outputs = model(sentences)
            preds = torch.argmax(outputs, dim=-1)
            mask = tags != PAD_IDX
            correct += (preds[mask] == tags[mask]).sum().item()
            total += mask.sum().item()
    return correct / total

# Huấn luyện
num_epochs = 5
for epoch in range(1, num_epochs+1):
    train_loss = train_epoch(model, train_loader)
    train_acc = evaluate(model, train_loader)
    dev_acc = evaluate(model, dev_loader)
    print(f"Epoch {epoch}: Loss={train_loss:.4f}, Train Acc={train_acc:.4f}, Dev Acc={dev_acc:.4f}")


Epoch 1: Loss=1.1047, Train Acc=0.7851, Dev Acc=0.7541
Epoch 2: Loss=0.6036, Train Acc=0.8486, Dev Acc=0.8036
Epoch 3: Loss=0.4511, Train Acc=0.8850, Dev Acc=0.8296
Epoch 4: Loss=0.3547, Train Acc=0.9087, Dev Acc=0.8448
Epoch 5: Loss=0.2860, Train Acc=0.9272, Dev Acc=0.8517


In [33]:
def predict_sentence(model, sentence):
    model.eval()
    tokens = sentence.strip().split()
    indices = [word_to_ix.get(w, word_to_ix["<UNK>"]) for w in tokens]
    inputs = torch.tensor(indices, dtype=torch.long).unsqueeze(0).to(device)
    with torch.no_grad():
        outputs = model(inputs)
        preds = torch.argmax(outputs, dim=-1).squeeze(0)
    idx_to_tag = {i: t for t, i in tag_to_ix.items()}
    return list(zip(tokens, [idx_to_tag[i.item()] for i in preds]))

# Ví dụ
predict_sentence(model, "I love NLP")


[('I', 'PRON'), ('love', 'VERB'), ('NLP', '_')]

In [48]:
import pandas as pd

# Đọc dữ liệu train/val/test
df_train = pd.read_csv('../hwu/train_10.csv', header=0)
df_val = pd.read_csv('../hwu/val.csv',header=0)
df_test = pd.read_csv('../hwu/test.csv', header=0)

print("Train shape:", df_train.shape)
print("Validation shape:", df_val.shape)
print("Test shape:", df_test.shape)

df_train.head()

Train shape: (640, 2)
Validation shape: (1076, 2)
Test shape: (1076, 2)


,text,category
0,remind me about my alarms today,alarm_query
1,list my different alarm,alarm_query
2,what alarms are set,alarm_query
3,list alarms,alarm_query
4,what's the alarm situation for tomorrow,alarm_query


In [50]:
from sklearn.preprocessing import LabelEncoder


# Chuyển intent sang dạng số
label_encoder = LabelEncoder()
label_encoder.fit(df_train['category'])


y_train = label_encoder.transform(df_train['category'])
y_val = label_encoder.transform(df_val['category'])
y_test = label_encoder.transform(df_test['category'])

In [51]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report


# 1. Tạo pipeline
tfidf_lr_pipeline = make_pipeline(
TfidfVectorizer(max_features=5000),
LogisticRegression(max_iter=1000)
)


# 2. Huấn luyện trên tập train
tfidf_lr_pipeline.fit(df_train['text'], y_train)


# 3. Đánh giá trên tập test
y_pred = tfidf_lr_pipeline.predict(df_test['text'])
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

                          precision    recall  f1-score   support

             alarm_query       0.67      0.74      0.70        19
            alarm_remove       0.25      0.09      0.13        11
               alarm_set       0.50      0.84      0.63        19
       audio_volume_down       0.44      0.88      0.58         8
       audio_volume_mute       0.65      0.73      0.69        15
         audio_volume_up       0.67      0.46      0.55        13
          calendar_query       0.31      0.26      0.29        19
         calendar_remove       0.88      0.79      0.83        19
            calendar_set       0.54      0.37      0.44        19
          cooking_recipe       0.50      0.26      0.34        19
        datetime_convert       0.40      0.75      0.52         8
          datetime_query       0.50      0.47      0.49        19
        email_addcontact       0.36      1.00      0.53         8
             email_query       0.81      0.68      0.74        19
      ema

In [52]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report


# 1. Tạo pipeline
tfidf_lr_pipeline = make_pipeline(
TfidfVectorizer(max_features=5000),
LogisticRegression(max_iter=1000)
)


# 2. Huấn luyện trên tập train
tfidf_lr_pipeline.fit(df_train['text'], y_train)


# 3. Đánh giá trên tập test
y_pred = tfidf_lr_pipeline.predict(df_test['text'])
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

                          precision    recall  f1-score   support

             alarm_query       0.67      0.74      0.70        19
            alarm_remove       0.25      0.09      0.13        11
               alarm_set       0.50      0.84      0.63        19
       audio_volume_down       0.44      0.88      0.58         8
       audio_volume_mute       0.65      0.73      0.69        15
         audio_volume_up       0.67      0.46      0.55        13
          calendar_query       0.31      0.26      0.29        19
         calendar_remove       0.88      0.79      0.83        19
            calendar_set       0.54      0.37      0.44        19
          cooking_recipe       0.50      0.26      0.34        19
        datetime_convert       0.40      0.75      0.52         8
          datetime_query       0.50      0.47      0.49        19
        email_addcontact       0.36      1.00      0.53         8
             email_query       0.81      0.68      0.74        19
      ema

In [55]:
import numpy as np
from gensim.models import Word2Vec
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout


# 1. Huấn luyện Word2Vec
sentences = [text.split() for text in df_train['text']]
w2v_model = Word2Vec(sentences, vector_size=100, window=5, min_count=1, workers=4)


# 2. Hàm chuyển câu thành vector trung bình
def sentence_to_avg_vector(text, model):
    words = text.split()
    vecs = [model.wv[w] for w in words if w in model.wv]
    if len(vecs) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vecs, axis=0)


# 3. Chuyển dữ liệu sang dạng vector trung bình
X_train_avg = np.array([sentence_to_avg_vector(text, w2v_model) for text in df_train['text']])
X_val_avg = np.array([sentence_to_avg_vector(text, w2v_model) for text in df_val['text']])
X_test_avg = np.array([sentence_to_avg_vector(text, w2v_model) for text in df_test['text']])


# 4. Xây dựng mô hình Dense
num_classes = len(label_encoder.classes_)
model_avg_dense = Sequential([
Dense(128, activation='relu', input_shape=(w2v_model.vector_size,)),
Dropout(0.5),
Dense(num_classes, activation='softmax')
])


# 5. Compile và huấn luyện
model_avg_dense.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model_avg_dense.fit(X_train_avg, y_train, validation_data=(X_val_avg, y_val), epochs=10, batch_size=32)

/Users/mac/Desktop/nlp-learner/venv312/lib/python3.12/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - accuracy: 0.0109 - loss: 4.1599 - val_accuracy: 0.0139 - val_loss: 4.1586
Epoch 2/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.0141 - loss: 4.1590 - val_accuracy: 0.0260 - val_loss: 4.1580
Epoch 3/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.0203 - loss: 4.1580 - val_accuracy: 0.0251 - val_loss: 4.1575
Epoch 4/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.0203 - loss: 4.1572 - val_accuracy: 0.0316 - val_loss: 4.1570
Epoch 5/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.0172 - loss: 4.1561 - val_accuracy: 0.0483 - val_loss: 4.1565
Epoch 6/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.0312 - loss: 4.1550 - val_accuracy: 0.0390 - val_loss: 4.1558
Epoch 7/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.0406 - loss: 4.1543 - val_accuracy: 0.0511 - val_loss: 4.1547
Epoch 8/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.0281 - loss: 4.1537 - val_accuracy: 0.0400 - val_l

In [57]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, LSTM


# 1. Tokenizer và padding
vocab_size = 5000
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<UNK>")
tokenizer.fit_on_texts(df_train['text'])


max_len = 50
X_train_seq = pad_sequences(tokenizer.texts_to_sequences(df_train['text']), maxlen=max_len, padding='post')
X_val_seq = pad_sequences(tokenizer.texts_to_sequences(df_val['text']), maxlen=max_len, padding='post')
X_test_seq = pad_sequences(tokenizer.texts_to_sequences(df_test['text']), maxlen=max_len, padding='post')


# 2. Tạo ma trận trọng số từ Word2Vec
embedding_dim = w2v_model.vector_size
embedding_matrix = np.zeros((vocab_size+1, embedding_dim))
for word, i in tokenizer.word_index.items():
    if i <= vocab_size and word in w2v_model.wv:
        embedding_matrix[i] = w2v_model.wv[word]


# 3. Xây dựng mô hình LSTM với Embedding pre-trained
lstm_model_pretrained = Sequential([
Embedding(input_dim=vocab_size+1, output_dim=embedding_dim, weights=[embedding_matrix], input_length=max_len, trainable=False),
LSTM(128, dropout=0.2, recurrent_dropout=0.2),
Dense(num_classes, activation='softmax')
])


# 4. Compile và huấn luyện
lstm_model_pretrained.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

/Users/mac/Desktop/nlp-learner/venv312/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [58]:
# 1. Xây dựng mô hình LSTM với Embedding học từ đầu
lstm_model_scratch = Sequential([
Embedding(input_dim=vocab_size+1, output_dim=100, input_length=max_len),
LSTM(128, dropout=0.2, recurrent_dropout=0.2),
Dense(num_classes, activation='softmax')
])


# 2. Compile và huấn luyện
lstm_model_scratch.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [59]:
from sklearn.metrics import f1_score
import numpy as np

# 1. TF-IDF + Logistic Regression
y_pred_tfidf = tfidf_lr_pipeline.predict(df_test['text'])
f1_tfidf = f1_score(y_test, y_pred_tfidf, average='macro')
# Test loss với Logistic Regression không áp dụng trực tiếp
loss_tfidf = "N/A"

# 2. Word2Vec (Avg) + Dense
y_pred_avg_dense_prob = model_avg_dense.predict(X_test_avg)
y_pred_avg_dense = np.argmax(y_pred_avg_dense_prob, axis=1)
f1_avg_dense = f1_score(y_test, y_pred_avg_dense, average='macro')
# Tính test loss
loss_avg_dense = model_avg_dense.evaluate(X_test_avg, y_test, verbose=0)[0]

# 3. Embedding Pre-trained + LSTM
y_pred_pretrained_prob = lstm_model_pretrained.predict(X_test_seq)
y_pred_pretrained = np.argmax(y_pred_pretrained_prob, axis=1)
f1_pretrained = f1_score(y_test, y_pred_pretrained, average='macro')
loss_pretrained = lstm_model_pretrained.evaluate(X_test_seq, y_test, verbose=0)[0]

# 4. Embedding Scratch + LSTM
y_pred_scratch_prob = lstm_model_scratch.predict(X_test_seq)
y_pred_scratch = np.argmax(y_pred_scratch_prob, axis=1)
f1_scratch = f1_score(y_test, y_pred_scratch, average='macro')
loss_scratch = lstm_model_scratch.evaluate(X_test_seq, y_test, verbose=0)[0]

# In bảng kết quả
import pandas as pd
df_results = pd.DataFrame({
    "Pipeline": [
        "TF-IDF + Logistic Regression",
        "Word2Vec (Avg) + Dense",
        "Embedding (Pre-trained) + LSTM",
        "Embedding (Scratch) + LSTM"
    ],
    "F1-score (Macro)": [f1_tfidf, f1_avg_dense, f1_pretrained, f1_scratch],
    "Test Loss": [loss_tfidf, loss_avg_dense, loss_pretrained, loss_scratch]
})
print(df_results)


34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
34/34 ━━━━━━━━━━━━━━━━━━━━ 2s 32ms/step
34/34 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step
                         Pipeline  F1-score (Macro) Test Loss
0    TF-IDF + Logistic Regression          0.642381       N/A
1          Word2Vec (Avg) + Dense          0.022756  4.152387
2  Embedding (Pre-trained) + LSTM          0.004813  4.158883
3      Embedding (Scratch) + LSTM          0.000542  4.159305


In [60]:
# Danh sách câu cần test
test_sentences = [
    "can you remind me to not call my mom",
    "is it going to be sunny or rainy tomorrow",
    "find a flight from new york to london but not through paris"
]

# Dự đoán
y_pred = tfidf_lr_pipeline.predict(test_sentences)

# Chuyển nhãn số về nhãn gốc
predicted_intents = label_encoder.inverse_transform(y_pred)

# Hiển thị kết quả
for text, intent in zip(test_sentences, predicted_intents):
    print(f"Text: {text}\nPredicted intent: {intent}\n")


Text: can you remind me to not call my mom
Predicted intent: calendar_set

Text: is it going to be sunny or rainy tomorrow
Predicted intent: general_negate

Text: find a flight from new york to london but not through paris
Predicted intent: general_negate

